# Get to Know a Dataset: Liars' Bench Expanded

This notebook is a guided introduction to the [Liars' Bench Expanded](https://registry.opendata.aws/liars-bench-expanded) dataset, hosted on the [Registry of Open Data on AWS](https://registry.opendata.aws/).

**Liars' Bench Expanded** pairs the labeled deception transcripts from [Liars' Bench](https://arxiv.org/abs/2511.16035) (Kretschmar et al., 2025) with pre-computed residual-stream activation tensors extracted from open-weight LLMs at inference time via the [National Deep Inference Fabric (NDIF)](https://ndif.us). The result is a dataset purpose-built for white-box deception detection: training linear probes directly on a model's internal representations to distinguish honest from deceptive outputs, without needing GPU access or re-running inference.

By the end of this notebook you will know:
- How the dataset is organized in S3
- How to load activation tensors and transcript metadata
- What the data looks like, and how to visualize it
- How to train a simple linear probe and evaluate it across transformer layers

## Setup

This notebook requires the following libraries. Install them with pip if needed:

```
zarr >= 3.0
s3fs >= 2024.0
pandas >= 2.0
pyarrow >= 15.0
numpy >= 1.26
scikit-learn >= 1.4
matplotlib >= 3.8
```

In [ ]:
import boto3
import numpy as np
import pandas as pd
import zarr
import s3fs
import matplotlib.pyplot as plt
from botocore import UNSIGNED
from botocore.config import Config
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.decomposition import PCA

# S3 bucket name for this dataset
BUCKET = "liars-bench-expanded"

# Anonymous S3 client for public bucket access
s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))
fs = s3fs.S3FileSystem(anon=True)

In [ ]:
# ── LOCAL TESTING CELL ──────────────────────────────────────────────────────
# Run this cell instead of the S3 loading cells (dd7f4bcf and ed25fd6e) to
# test the notebook against the sample files included in this repository.
# Remove or skip this cell when running against the real S3 bucket.

import json

# Load activations.npy → in-memory Zarr array
acts_np = np.load("activations.npy")          # shape: (20, 32, 4096), float16
store = zarr.array(acts_np)                    # wraps numpy array as zarr

# Load meta.jsonl → DataFrame matching the notebook's expected schema
records = [json.loads(l) for l in open("meta.jsonl")]
df = pd.DataFrame(records)

# Rename generator_model → model to match Liars' Bench schema
df = df.rename(columns={"generator_model": "model", "dataset": "source_dataset"})

# Reconstruct a minimal messages field from prompt_text / completion_text
df["messages"] = df.apply(
    lambda r: [{"role": "user",      "content": r["prompt_text"]},
               {"role": "assistant", "content": r["completion_text"]}],
    axis=1
)

# The sample has all deceptive=True. Flip the last 10 records to False so
# label-distribution and probe-training cells have both classes to work with.
# NOTE: these flipped labels are synthetic — for testing code paths only.
df.loc[df.index[10:], "deceptive"] = False

labels = df["deceptive"].values
print(f"Records: {len(df)}  |  deceptive: {labels.sum()}  |  honest: {(~labels).sum()}")
print(f"Store shape: {store.shape}  (n_records, n_layers, hidden_dim)")

---

### Q: How is the dataset organized? Help us understand the S3 bucket structure.

The bucket has two top-level prefixes: `data/` for activation tensors and `metadata/` for transcript labels. Both use [Hive-style partitioning](https://docs.aws.amazon.com/athena/latest/ug/partitions.html) — `key=value` path segments — so you can navigate directly to the slice you need without listing the full bucket.

Let's start by listing the top-level prefixes.

In [ ]:
response = s3.list_objects_v2(Bucket=BUCKET, Delimiter="/")
for prefix in response.get("CommonPrefixes", []):
    print(prefix["Prefix"])

The first partition level under both prefixes is `captured_model=` — the model whose internal activations were recorded. This is the natural unit for probe training, since a probe is always model-specific. Let's list the available models.

In [ ]:
response = s3.list_objects_v2(Bucket=BUCKET, Prefix="data/", Delimiter="/")
for prefix in response.get("CommonPrefixes", []):
    print(prefix["Prefix"])

Under each `captured_model=` prefix is a `scenario=` level, which encodes the `(generator_model, source_dataset, year)` combination that produced those transcripts. Let's look at the scenarios available for one model.

In [ ]:
MODEL = "qwen3.5-9b"
prefix = f"data/captured_model={MODEL}/"

response = s3.list_objects_v2(Bucket=BUCKET, Prefix=prefix, Delimiter="/")
for p in response.get("CommonPrefixes", []):
    print(p["Prefix"])

Each scenario contains a `split=` level (e.g. `split=test`), and within each split lives a single `activations.zarr/` store. The full path to any activation array follows the pattern:

```
s3://liars-bench-expanded/data/captured_model={model}/scenario={scenario}/split={split}/activations.zarr/
```

And its matching metadata Parquet file:

```
s3://liars-bench-expanded/metadata/captured_model={model}/scenario={scenario}/split={split}/records.parquet
```

---

### Q: What data formats are used? Why?

The dataset uses two formats: **Zarr v3** for activation tensors and **Parquet** for transcript metadata.

**Zarr v3** is a chunked, compressed array format designed for cloud storage. Each activation array has shape `(n_records, n_layers, hidden_dim)` and is stored with chunk shape `(1000, 1, 4096)` — one chunk spans 1000 records and exactly one layer. This means fetching all records for a single layer (the primary access pattern for probe training) downloads only the chunks for that layer, in parallel, without touching the rest of the array. At 1000 records per chunk and ~8 MB per chunk, a full layer read for 72k records takes ~73 HTTP requests rather than one 600 MB download. See the [Zarr documentation](https://zarr.readthedocs.io/en/stable/) for more.

**Parquet** is a columnar binary format well-suited for structured tabular data. It supports predicate pushdown and column projection, so you can load just the `deceptive` and `index` columns without reading `messages` or `prompt_text`. Parquet is natively supported by pandas, PyArrow, Polars, DuckDB, and Amazon Athena. See the [Apache Parquet documentation](https://parquet.apache.org/docs/) for more.

---

### Q: Can you show us an example of loading data from the dataset?

Let's load the metadata for one scenario and split, then open the corresponding activation array.

In [ ]:
MODEL    = "qwen3.5-9b"
SCENARIO = "qwen-2.5-72b-instruct_wmdp-bio-2026"
SPLIT    = "test"

# Load transcript metadata from Parquet
meta_path = f"s3://{BUCKET}/metadata/captured_model={MODEL}/scenario={SCENARIO}/split={SPLIT}/records.parquet"
df = pd.read_parquet(meta_path, filesystem=fs)

print(f"Records: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head()

Each row is one transcript. The `deceptive` column is our label. The `messages` column contains the full conversation in `[{"role": ..., "content": ...}]` format, preserving the original Liars' Bench structure. Let's look at one record's conversation alongside its label.

In [ ]:
row = df.iloc[0]
print(f"deceptive: {row['deceptive']}")
print(f"model: {row['model']}")
print(f"seq_len: {row['seq_len']}")
print()
for turn in row["messages"]:
    print(f"[{turn['role']}]")
    print(turn["content"][:300])
    print()

Now let's open the corresponding Zarr activation array. The array has shape `(n_records, n_layers, hidden_dim)`. Opening it is instantaneous — Zarr reads only the metadata header until we actually index into the array.

In [ ]:
zarr_path = f"s3://{BUCKET}/data/captured_model={MODEL}/scenario={SCENARIO}/split={SPLIT}/activations.zarr"
store = zarr.open(zarr_path, mode="r", storage_options={"anon": True})

print(f"Shape:  {store.shape}   # (n_records, n_layers, hidden_dim)")
print(f"Dtype:  {store.dtype}")
print(f"Chunks: {store.chunks} # (records_per_chunk, layers_per_chunk, hidden_dim)")

To fetch activations for a single layer across all records, index the second dimension. Zarr downloads only the chunks for that layer — no other data is transferred.

In [ ]:
LAYER = 15
activations_layer15 = store[:, LAYER, :]   # shape: (n_records, hidden_dim)

print(f"Layer {LAYER} activations: {activations_layer15.shape}")
print(f"Memory: {activations_layer15.nbytes / 1e6:.1f} MB")

---

### Q: Show us some visuals that illustrate the dataset.

Let's start with the distribution of honest vs. deceptive labels across scenarios.

In [ ]:
label_counts = df.groupby(["deceptive"]).size().reset_index(name="count")
label_counts["label"] = label_counts["deceptive"].map({True: "Deceptive", False: "Honest"})

fig, ax = plt.subplots(figsize=(5, 4))
colors = ["#e74c3c", "#2ecc71"]
ax.bar(label_counts["label"], label_counts["count"], color=colors, edgecolor="white")
ax.set_title(f"Label distribution\n({MODEL} · {SCENARIO})", fontsize=11)
ax.set_ylabel("Records")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

Now let's look at how activation magnitudes (L2 norm) vary across layers, split by deception label. If deceptive and honest responses produce systematically different norms at certain layers, that's a signal that those layers carry information about the model's internal state during deception.

In [ ]:
labels = df["deceptive"].values
n_layers = store.shape[1]

mean_norms = {True: [], False: []}

for layer_idx in range(n_layers):
    acts = store[:, layer_idx, :].astype(np.float32)   # (n_records, hidden_dim)
    norms = np.linalg.norm(acts, axis=1)               # (n_records,)
    for label in [True, False]:
        mean_norms[label].append(norms[labels == label].mean())

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(mean_norms[True],  label="Deceptive", color="#e74c3c", linewidth=2)
ax.plot(mean_norms[False], label="Honest",    color="#2ecc71", linewidth=2)
ax.set_xlabel("Layer")
ax.set_ylabel("Mean L2 norm")
ax.set_title("Residual-stream activation magnitude by layer")
ax.legend()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

Let's also project one layer's activations down to 2D with PCA to visually inspect whether deceptive and honest records separate in activation space.

In [ ]:
acts = store[:, LAYER, :].astype(np.float32)
pca = PCA(n_components=2)
coords = pca.fit_transform(acts)

fig, ax = plt.subplots(figsize=(6, 5))
for label, color, name in [(True, "#e74c3c", "Deceptive"), (False, "#2ecc71", "Honest")]:
    mask = labels == label
    ax.scatter(coords[mask, 0], coords[mask, 1], c=color, label=name, alpha=0.5, s=15, edgecolors="none")

ax.set_title(f"PCA of layer {LAYER} activations")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)")
ax.legend()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

---

### Q: What is one question you have answered using this data?

**Which transformer layers carry the most information about deception?**

A linear probe is a logistic regression classifier trained on a single layer's activations to predict the `deceptive` label. If a probe trained on layer *N* achieves high accuracy, it means that layer's residual stream linearly encodes information that distinguishes honest from deceptive outputs.

We train one probe per layer using 5-fold cross-validation and plot accuracy across all layers. Layers where accuracy peaks are the most informative for deception detection.

In [ ]:
probe_accuracies = []

for layer_idx in range(n_layers):
    acts = store[:, layer_idx, :].astype(np.float32)
    clf = LogisticRegression(max_iter=1000, C=0.1)
    scores = cross_val_score(clf, acts, labels, cv=5, scoring="accuracy")
    probe_accuracies.append(scores.mean())
    print(f"Layer {layer_idx:2d}: {scores.mean():.3f} ± {scores.std():.3f}")

In [ ]:
best_layer = int(np.argmax(probe_accuracies))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(probe_accuracies, color="#3498db", linewidth=2, marker="o", markersize=4)
ax.axvline(best_layer, color="#e74c3c", linestyle="--", linewidth=1.5,
           label=f"Best layer: {best_layer} ({probe_accuracies[best_layer]:.3f})")
ax.axhline(0.5, color="gray", linestyle=":", linewidth=1, label="Chance")
ax.set_xlabel("Layer")
ax.set_ylabel("Probe accuracy (5-fold CV)")
ax.set_title("Linear probe accuracy by layer")
ax.set_ylim(0.4, 1.05)
ax.legend()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

print(f"\nBest layer: {best_layer}  (accuracy: {probe_accuracies[best_layer]:.3f})")

The probe accuracy curve shows where in the model deception is most linearly detectable. Early layers tend to encode surface-level token features; middle-to-late layers are where higher-level semantic representations emerge and where probes typically peak. The exact pattern varies by model and scenario — exploring this across the full dataset is one of the core research questions this dataset was built to support.

---

### Q: What is one unanswered question you think could be answered with this data?

**Do deception probes generalize across scenarios and models?**

The example above trains a probe on one scenario (`qwen-2.5-72b-instruct_wmdp-bio-2026`) and evaluates within that scenario. A deeper and more practically important question is: *can a probe trained on one deception scenario detect deception in a completely different one?*

For example:
- Train on `alpaca` (instruction-following deception), evaluate on `insider-trading` (financial deception)
- Train on `Qwen3.5-9B` activations, evaluate on `Llama-3-8B` activations for the same scenario

If probes generalize across scenarios, it suggests the model has a shared internal representation of deception — a "deception direction" in activation space — rather than scenario-specific features. If they don't generalize, it suggests deception is encoded differently depending on context, with implications for how robust any deployed lie detector could be.

**To get started:** the dataset structure makes this straightforward. Load `records.parquet` and `activations.zarr` for two different scenarios sharing the same `captured_model`, train a probe on one, and evaluate it on the other using the `predict` method of your fitted `LogisticRegression`. The `scenario=` partition structure means you never need to filter — just point at a different path.

---

## Further resources

- [Dataset landing page](https://registry.opendata.aws/liars-bench-expanded) — Registry of Open Data on AWS
- [Documentation & data dictionary](https://github.com/ndif-team/liars-bench-expanded)
- [Probe training tutorial](https://github.com/ndif-team/open-data-examples/blob/main/liars-bench-expanded/training-probes.ipynb) — a deeper dive into probe training methodology
- [Original Liars' Bench paper](https://arxiv.org/abs/2511.16035) — Kretschmar et al., 2025
- [NNsight](https://nnsight.net) — the library used to extract activations via NDIF
- Questions? Post on [AWS re:Post](https://repost.aws/tags/TApd0Wl5P8S9O6riTWmh-cGw/aws-open-data) or open an issue on [GitHub](https://github.com/ndif-team/liars-bench-expanded/issues)

---
*Please clear all cell outputs before committing this notebook to GitHub.*